In [1]:
import sys
sys.path.append("..")          # pour trouver le dossier src/

import numpy as np
from src.data import load_raw, build
from src.utils import evaluate

X_tr, Y_tr, X_te, Y_te, A = load_raw()
print(X_tr.shape, Y_tr.shape, X_te.shape, Y_te.shape, A.shape)

(1261, 36, 48) (1261, 36) (840, 36, 48) (840, 36) (36, 36)


In [2]:
for k in (0, 1, 2):
    corr = [np.corrcoef(X_tr[k:, :, j].ravel(), Y_tr[:len(Y_tr) - k].ravel())[0, 1] for j in range(48)]
    corr = np.nan_to_num(corr)
    j = int(np.argmax(corr)); print(f"décalage {k} : colonne {j}, corrélation {corr[j]:.5f}")

décalage 0 : colonne 9, corrélation 0.96227
décalage 1 : colonne 9, corrélation 0.98103
décalage 2 : colonne 9, corrélation 1.00000


In [3]:
print(np.allclose(X_tr[2:, :, 9], Y_tr[:-2]))   # doit afficher True
for j in range(10):
    for k in range(1, 20):
        if np.allclose(X_tr[k:, :, j], Y_tr[:-k]):
            print(f"colonne {j} = volume {k} quarts d'heure avant la cible")

d = build("base")
print(evaluate(d["y_val"], d["X_val"][:, 9]))

True
colonne 0 = volume 11 quarts d'heure avant la cible
colonne 1 = volume 10 quarts d'heure avant la cible
colonne 2 = volume 9 quarts d'heure avant la cible
colonne 3 = volume 8 quarts d'heure avant la cible
colonne 4 = volume 7 quarts d'heure avant la cible
colonne 5 = volume 6 quarts d'heure avant la cible
colonne 6 = volume 5 quarts d'heure avant la cible
colonne 7 = volume 4 quarts d'heure avant la cible
colonne 8 = volume 3 quarts d'heure avant la cible
colonne 9 = volume 2 quarts d'heure avant la cible
{'MAE': 0.03661767441898628, 'RMSE': 0.05589856254155216}


In [4]:
Xf = X_tr.reshape(-1, 48)       # on aplatit instants × capteurs
for j in range(10, 48):
    u = np.unique(Xf[:, j])
    print(j, len(u), u[:4])

10 2 [0. 1.]
11 2 [0. 1.]
12 2 [0. 1.]
13 2 [0. 1.]
14 2 [0. 1.]
15 2 [0. 1.]
16 2 [0. 1.]
17 2 [0. 1.]
18 2 [0. 1.]
19 2 [0. 1.]
20 2 [0. 1.]
21 2 [0. 1.]
22 2 [0. 1.]
23 2 [0. 1.]
24 2 [0. 1.]
25 2 [0. 1.]
26 2 [0. 1.]
27 2 [0. 1.]
28 2 [0. 1.]
29 2 [0. 1.]
30 2 [0. 1.]
31 2 [0. 1.]
32 2 [0. 1.]
33 2 [0. 1.]
34 2 [0. 1.]
35 2 [0. 1.]
36 2 [0. 1.]
37 2 [0. 1.]
38 2 [0. 1.]
39 5 [1. 2. 3. 4.]
40 2 [0. 1.]
41 2 [0. 1.]
42 2 [0. 1.]
43 2 [0. 1.]
44 2 [0. 1.]
45 2 [0. 1.]
46 2 [0. 1.]
47 2 [0. 1.]


In [5]:
for a, b, nom in [(10, 17, "jour ?"), (17, 41, "heure ?"), (41, 45, "direction ?")]:
    print(nom, np.unique(Xf[:, a:b].sum(1)))     # attendu : [1.]

jour ? [1. 2.]
heure ? [1. 2. 3. 4. 5. 6. 7.]
direction ? [0. 1. 2.]


In [6]:
Xf = X_tr.reshape(-1, 48)
bin_cols = [j for j in range(10, 48) if set(np.unique(Xf[:, j])) <= {0., 1.}]

blocks = []
for a in range(10, 48):
    for b in range(a + 2, 49):
        if all(c in bin_cols for c in range(a, b)) and (Xf[:, a:b].sum(1) == 1).all():
            blocks.append((a, b))
print(blocks)

static = [j for j in range(10, 48)
          if all(len(np.unique(X_tr[:, s, j])) == 1 for s in range(36))]
print("colonnes fixes par capteur :", static)
print("colonnes qui varient dans le temps :", [j for j in range(10, 48) if j not in static])

print(np.unique(X_tr[0, :, 39], return_counts=True))

[(10, 15), (15, 39), (40, 44), (43, 46), (44, 48)]
colonnes fixes par capteur : [39, 40, 41, 42, 43, 44, 45, 46, 47]
colonnes qui varient dans le temps : [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]
(array([1., 2., 3., 4., 5.]), array([ 1,  3, 10, 20,  2]))


In [7]:
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

cols = ([f"lag_{k}" for k in range(11, 1, -1)]      # colonnes 0 à 9
        + [f"jour_{i}" for i in range(5)]           # 10 à 14
        + [f"heure_{h}" for h in range(24)]         # 15 à 38
        + ["voies"]                                 # 39
        + [f"bloc40_{i}" for i in range(4)]         # 40 à 43
        + [f"bloc44_{i}" for i in range(4)])        # 44 à 47
assert len(cols) == 48

Xf = X_tr.reshape(-1, 48)
df = pd.DataFrame(Xf, columns=cols)
df["instant"] = np.repeat(np.arange(len(X_tr)), 36)
df["capteur"] = np.tile(np.arange(36), len(X_tr))
df["jour"] = Xf[:, 10:15].argmax(1)                 # 0 à 4 (version lisible)
df["heure"] = Xf[:, 15:39].argmax(1)                # 0 à 23
df["y"] = Y_tr.reshape(-1)

print(df.shape)                                     # attendu : (45396, 56)

(45396, 53)


In [8]:
df[df.capteur == 0][["instant", "jour", "heure", "lag_3", "lag_2", "y"]].head(15)

,instant,jour,heure,lag_3,lag_2,y
0,0,0,0,0.054647,0.050911,0.050911
36,1,0,0,0.050911,0.042971,0.047641
72,2,0,0,0.042971,0.050911,0.044839
108,3,0,0,0.050911,0.047641,0.059785
144,4,0,1,0.047641,0.044839,0.062588
180,5,0,1,0.044839,0.059785,0.063989
216,6,0,1,0.059785,0.062588,0.088277
252,7,0,1,0.062588,0.063989,0.113498
288,8,0,2,0.063989,0.088277,0.122840
324,9,0,2,0.088277,0.113498,0.125175
